In [ ]:
!pip install implicit -qqq
!pip install git+https://github.com/daviddavo/lightfm -qqq
!pip install rarfile -qqq

  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd
import numpy as np
import implicit
import ast
from scipy.sparse import csr_matrix
from implicit.evaluation import train_test_split, precision_at_k, ndcg_at_k
from lightfm import LightFM
from lightfm.datasets import fetch_movielens
from lightfm.evaluation import precision_at_k as lightfm_precision_at_k
from lightfm.evaluation import auc_score as lightfm_auc_score

In [ ]:
import gdown, zipfile
from rarfile import RarFile
url = 'https://drive.google.com/file/d/1sSJvYvSj2jmylAUBlMwmPAd2RMeEbpwl/view?usp=sharing'
path = gdown.download(url, quiet=False, fuzzy=True)
#with zipfile.ZipFile(path, 'r') as f: f.extractall('')
with RarFile(path) as f:f.extractall('')

Downloading...
From: https://drive.google.com/uc?id=1sSJvYvSj2jmylAUBlMwmPAd2RMeEbpwl
To: /content/Retail.rar
100%|██████████| 29.8M/29.8M [00:00<00:00, 34.7MB/s]


#Часть 1

##1. Подготовка данных:

Загрузите набор данных и подготовьте три компонента:

‒ матрицу взаимодействий;

‒ признаки пользователей;

‒ признаки товаров.

In [ ]:
print("Задание 1")
retail_dataframe = pd.read_csv('Retail.csv')
retail_dataframe['Product'] = retail_dataframe['Product'].apply(ast.literal_eval)
retail_dataframe = retail_dataframe.explode('Product')

retail_dataframe['date_time_transformed'] = pd.to_datetime(retail_dataframe['Date'])
retail_dataframe['day_of_week'] = retail_dataframe['date_time_transformed'].dt.dayofweek

user_features_aggregated = retail_dataframe.groupby('Customer_Name').agg(
    unique_transactions_count=('Transaction_ID', 'nunique'),
    total_items_sum=('Total_Items', 'sum')
)

item_features_aggregated = retail_dataframe.groupby('Product').agg(
    unique_customers_count=('Customer_Name', 'count'),
    average_day_of_week=('day_of_week', 'mean')
)

user_categorical_codes = retail_dataframe['Customer_Name'].astype('category').cat.codes
item_categorical_codes = retail_dataframe['Product'].astype('category').cat.codes

interaction_matrix = csr_matrix(
    (np.ones(len(retail_dataframe), dtype='float32'),
    (user_categorical_codes, item_categorical_codes))
)

train_interactions, test_interactions = train_test_split(interaction_matrix, train_percentage=0.8)

class SparseMatrixWithMemoryViewFix(csr_matrix):
    def __getitem__(self, index_key):
        if type(index_key).__name__ == '_memoryviewslice':
            return super().__getitem__(np.asarray(index_key))
        return super().__getitem__(index_key)

train_interactions = SparseMatrixWithMemoryViewFix(train_interactions)
test_interactions = SparseMatrixWithMemoryViewFix(test_interactions)

Задание 1


In [ ]:
display(retail_dataframe.head())
display(user_features_aggregated.head())
display(item_features_aggregated.head())
display(user_categorical_codes.head())
display(item_categorical_codes.head())

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion,date_time_transformed,day_of_week
0,1000000000,2022-01-21 06:27:29,Stacey Price,Ketchup,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN,2022-01-21 06:27:29,4
0,1000000000,2022-01-21 06:27:29,Stacey Price,Shaving Cream,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN,2022-01-21 06:27:29,4
0,1000000000,2022-01-21 06:27:29,Stacey Price,Light Bulbs,3,71.65,Mobile Payment,Los Angeles,Warehouse Club,True,Homemaker,Winter,NaN,2022-01-21 06:27:29,4
1,1000000001,2023-03-01 13:01:21,Michelle Carlson,Ice Cream,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),2023-03-01 13:01:21,2
1,1000000001,2023-03-01 13:01:21,Michelle Carlson,Milk,2,25.93,Cash,San Francisco,Specialty Store,True,Professional,Fall,BOGO (Buy One Get One),2023-03-01 13:01:21,2


,unique_transactions_count,total_items_sum
Customer_Name,,
Aaron Acevedo,4,27
Aaron Acosta,3,42
Aaron Adams,12,210
Aaron Adkins,3,46
Aaron Aguilar,5,99


,unique_customers_count,average_day_of_week
Product,,
Air Freshener,36846,2.979781
Apple,36341,2.987672
BBQ Sauce,36626,3.020559
Baby Wipes,36794,2.976409
Banana,36813,3.009616


,0
0,291925
0,291925
0,291925
1,223107
1,223107


,0
0,37
0,60
0,40
1,32
1,42


##2. Обучение модели:

o Используйте алгоритмы ALS (Alternate Least Squares) и BRP для обучения модели.

In [ ]:
print("Задание 2")
print("ALS старт")

als_recommender_model = implicit.als.AlternatingLeastSquares(
    factors=50,
    iterations=15
)
als_recommender_model.fit(train_interactions)

print("ALS финиш; BPR старт")

bpr_recommender_model = implicit.bpr.BayesianPersonalizedRanking(
    factors=50,
    iterations=15
)
bpr_recommender_model.fit(train_interactions)

print("BPR финиш")

Задание 2
ALS старт


  0%|          | 0/15 [00:00<?, ?it/s]

ALS финиш; BPR старт


  0%|          | 0/15 [00:00<?, ?it/s]

BPR финиш


##3. Генерация рекомендаций:

‒ Создайте список топ-N рекомендаций для нескольких пользователей

‒ Для проверки выберите хотя бы 5 пользователей.

In [ ]:
print("Задание 3")

for current_user_id in range(5):
    recommendations_list = als_recommender_model.recommend(
        current_user_id,
        train_interactions[current_user_id],
        N=10
    )
    print(f"U{current_user_id}: {recommendations_list}")

Задание 3
U0: (array([48, 16, 39, 68,  9, 80, 19, 62, 41, 22], dtype=int32), array([0.34033266, 0.33441615, 0.29039362, 0.28549093, 0.27757636,
       0.26936483, 0.26804215, 0.25742134, 0.24991766, 0.24602333],
      dtype=float32))
U1: (array([31, 46, 67, 33, 62, 55, 74, 77, 68, 63], dtype=int32), array([0.32805374, 0.3187461 , 0.24847461, 0.2353924 , 0.22021051,
       0.21851245, 0.21452534, 0.21051453, 0.20107275, 0.19772083],
      dtype=float32))
U2: (array([72, 54,  5, 69, 42, 20, 43, 19, 61,  3], dtype=int32), array([0.55535275, 0.54405844, 0.51304495, 0.46296495, 0.42982244,
       0.38259065, 0.34601352, 0.34092754, 0.33921537, 0.3267724 ],
      dtype=float32))
U3: (array([75, 39, 45, 35, 17, 20, 18, 78,  4, 30], dtype=int32), array([0.33960587, 0.33140203, 0.29732254, 0.2688777 , 0.26108286,
       0.24918391, 0.23761779, 0.21409114, 0.20200567, 0.19939762],
      dtype=float32))
U4: (array([33, 72, 79, 24, 38, 17, 29, 34,  7, 71], dtype=int32), array([0.42180797, 0.372302

##4. Оценка модели:

‒ Оцените производительность модели, используя метрики NDCG, precision@k и AUC для разных моделей.

‒ Проверьте результаты как на обучающем, так и на тестовом наборах данных.

In [ ]:
from implicit.evaluation import precision_at_k, ndcg_at_k, AUC_at_k

print("Задание 4")

als_precision_score = precision_at_k(als_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)
als_ndcg_score = ndcg_at_k(als_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)
als_auc_score = AUC_at_k(als_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)

bpr_precision_score = precision_at_k(bpr_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)
bpr_ndcg_score = ndcg_at_k(bpr_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)
bpr_auc_score = AUC_at_k(bpr_recommender_model, train_interactions, test_interactions, K=10, show_progress=False)

print(f"ALS - P@10: {als_precision_score:.4f}, NDCG@10: {als_ndcg_score:.4f}, AUC@10: {als_auc_score:.4f}")
print(f"BPR - P@10: {bpr_precision_score:.4f}, NDCG@10: {bpr_ndcg_score:.4f}, AUC@10: {bpr_auc_score:.4f}")

Задание 4
ALS - P@10: 0.1665, NDCG@10: 0.0825, AUC@10: 0.5080
BPR - P@10: 0.1776, NDCG@10: 0.0920, AUC@10: 0.5148


#Часть 2

##1-3
1. Установить библиотеку LightFM

pip install lightfm

2. Выполнить импорт необходимых библиотек

from lightfm import LightFM

from lightfm.datasets import fetch_movielens # Для использования встроенного в LightFM набора данных Movielens

from lightfm.evaluation import precision_at_k, auc_score

3. Подготовка данных

Функция fetch_movielens загружает набор данных Movielens и возвращает словарь, содержащий матрицу оценок, пользовательские функции, характеристики элементов и другую информацию. В построении данной рекомендательной системы нас интересует только матрица оценок.

data = fetch_movielens(min_rating=5.0)

In [ ]:
print("Задания 1-3")

movielens_dataset = fetch_movielens(min_rating=5.0)

Задания 1-3


In [ ]:
display(list(movielens_dataset.items())[:10])

[('train',
  <COOrdinate sparse matrix of dtype 'int32'
  	with 19048 stored elements and shape (943, 1682)>),
 ('test',
  <COOrdinate sparse matrix of dtype 'int32'
  	with 2153 stored elements and shape (943, 1682)>),
 ('item_features',
  <Compressed Sparse Row sparse matrix of dtype 'float32'
  	with 1682 stored elements and shape (1682, 1682)>),
 ('item_feature_labels',
  array(['T', 'G', 'F', ..., 'S', 'Y', 'S'], dtype='<U1')),
 ('item_labels', array(['T', 'G', 'F', ..., 'S', 'Y', 'S'], dtype='<U1'))]

4. Реализация гибридной рекомендательной системы

Для реализации рекомендательной системы с lightFM предпочтительным алгоритмом является WARP (Weighted
Approximate-Rank Pairwise).

model = LightFM(loss='warp')

model.fit(data['train'], epochs=30, num_threads=2)

In [ ]:
print("Задание 4")

lightfm_recommender_model = LightFM(loss='warp')
best_observed_auc_score = 0.0
early_stopping_counter = 0
patience_threshold = 3

for epoch_index in range(30):
    lightfm_recommender_model.fit_partial(
        movielens_dataset['train'],
        epochs=1,
        num_threads=2
    )

    train_precision_metric = np.mean(lightfm_precision_at_k(lightfm_recommender_model, movielens_dataset['train'], k=10, num_threads=2))
    train_auc_metric = np.mean(lightfm_auc_score(lightfm_recommender_model, movielens_dataset['train'], num_threads=2))

    test_precision_metric = np.mean(lightfm_precision_at_k(lightfm_recommender_model, movielens_dataset['test'], k=10, num_threads=2))
    test_auc_metric = np.mean(lightfm_auc_score(lightfm_recommender_model, movielens_dataset['test'], num_threads=2))

    print(f"E:{epoch_index:02d} | Tr_P:{train_precision_metric:.4f} Tr_AUC:{train_auc_metric:.4f} | Te_P:{test_precision_metric:.4f} Te_AUC:{test_auc_metric:.4f}")

    if test_auc_metric > best_observed_auc_score:
        best_observed_auc_score = test_auc_metric
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience_threshold:
        print(f"Early stop. E:{epoch_index}, Max_Te_AUC:{best_observed_auc_score:.4f}")
        break

Задание 4
E:00 | Tr_P:0.2033 Tr_AUC:0.8894 | Te_P:0.0312 Te_AUC:0.8688
E:01 | Tr_P:0.2209 Tr_AUC:0.9047 | Te_P:0.0342 Te_AUC:0.8799
E:02 | Tr_P:0.2383 Tr_AUC:0.9148 | Te_P:0.0378 Te_AUC:0.8865
E:03 | Tr_P:0.2517 Tr_AUC:0.9227 | Te_P:0.0396 Te_AUC:0.8912
E:04 | Tr_P:0.2630 Tr_AUC:0.9291 | Te_P:0.0410 Te_AUC:0.8955
E:05 | Tr_P:0.2728 Tr_AUC:0.9342 | Te_P:0.0406 Te_AUC:0.8987
E:06 | Tr_P:0.2809 Tr_AUC:0.9382 | Te_P:0.0425 Te_AUC:0.9015
E:07 | Tr_P:0.2864 Tr_AUC:0.9416 | Te_P:0.0415 Te_AUC:0.9038
E:08 | Tr_P:0.2913 Tr_AUC:0.9447 | Te_P:0.0428 Te_AUC:0.9055
E:09 | Tr_P:0.2956 Tr_AUC:0.9474 | Te_P:0.0439 Te_AUC:0.9072
E:10 | Tr_P:0.3008 Tr_AUC:0.9498 | Te_P:0.0443 Te_AUC:0.9086
E:11 | Tr_P:0.3090 Tr_AUC:0.9520 | Te_P:0.0443 Te_AUC:0.9097
E:12 | Tr_P:0.3122 Tr_AUC:0.9539 | Te_P:0.0452 Te_AUC:0.9104
E:13 | Tr_P:0.3164 Tr_AUC:0.9556 | Te_P:0.0453 Te_AUC:0.9112
E:14 | Tr_P:0.3209 Tr_AUC:0.9571 | Te_P:0.0457 Te_AUC:0.9118
E:15 | Tr_P:0.3200 Tr_AUC:0.9586 | Te_P:0.0453 Te_AUC:0.9123
E:16 | Tr_P:0.

5. Генерация рекомендаций

Используем созданную модель для генерации рекомендаций для пользователей, вызвав метод recommend. Метод recommend
принимает два аргумента: user_ids и item_ids. Мы можем генерировать рекомендации для одного пользователя или
нескольких пользователей одновременно. Мы также можем указать количество рекомендаций для генерации.

Следующий код генерирует 10 рекомендаций для пользователя 3.

import numpy as np

user_id = 3

n_items = data['train'].shape[1]

recommendations = model.predict(user_id, np.arange(n_items))

top_items = np.argsort(-recommendations)[:10]

In [ ]:
print("Задание 5")

total_items_count = movielens_dataset['train'].shape[1]
prediction_scores = lightfm_recommender_model.predict(
    3,
    np.arange(total_items_count)
)

sorted_top_items_indices = np.argsort(-prediction_scores)[:10]
print(f"Top 10 U3: {sorted_top_items_indices}")

Задание 5
Top 10 U3: [326 323 358 880 878 305 241 288 345 299]


##6. Оценка рекомендательной системы
Оцените производительность модели, используя метрики precision@k и AUC. Метрика precision at k измеряет процент рекомендаций, которые были релевантны пользователю, из числа k лучших рекомендаций. Метрика AUC измеряет площадь под кривой рабочей характеристики приемника (ROC), которая показывает частоту истинных положительных срабатываний по сравнению с частотой ложных положительных срабатываний.

Мы можем использовать библиотеку LightFM для вычисления этих показателей. Точность из k рекомендаций может быть рассчитана с помощью метода precision@k, а AUC может быть рассчитан с помощью метода auc_score. В этом примере мы обучаем модель на основе матрицы взаимодействий и оцениваем точность по метрикам k и AUC как для обучающего, так и для тестового наборов данных. Мы используем значение 10 для k, что означает, что мы рассматриваем только 10 лучших рекомендаций для каждого пользователя.

Можно также попробовать вариант модели для k = 5.

#### Оцениваем модель на обучающих данных.

train_precision = np.mean(precision_at_k(model, data['train'], k=10,

num_threads=2))

train_auc = np.mean(auc_score(model, data['train'], num_threads=2))

#### Оцениваем модель на тестовых данных.

test_precision = np.mean(precision_at_k(model, data['test'], k=10,

num_threads=2))

test_auc = np.mean(auc_score(model, data['test'], num_threads=2))

print('Train precision: {:.2f}'.format(train_precision))

print('Train AUC: {:.2f}'.format(train_auc))

print('Test precision: {:.2f}'.format(test_precision)) print('Test AUC:

{:.2f}'.format(test_auc))

##7. Анализ результатов

Train precision: 0.51

Train AUC: 0.95

Test precision: 0.08

Test AUC: 0.92

Проанализируйте результат рекомендаций, сделайте выводы.


In [ ]:
print("Задания 6-7")

final_train_precision = np.mean(lightfm_precision_at_k(lightfm_recommender_model, movielens_dataset['train'], k=10, num_threads=2))
final_train_auc = np.mean(lightfm_auc_score(lightfm_recommender_model, movielens_dataset['train'], num_threads=2))

final_test_precision = np.mean(lightfm_precision_at_k(lightfm_recommender_model, movielens_dataset['test'], k=10, num_threads=2))
final_test_auc = np.mean(lightfm_auc_score(lightfm_recommender_model, movielens_dataset['test'], num_threads=2))

print('Train precision: {:.2f}'.format(final_train_precision))
print('Train AUC: {:.2f}'.format(final_train_auc))
print('Test precision: {:.2f}'.format(final_test_precision))
print('Test AUC: {:.2f}'.format(final_test_auc))

print("Вывод: метрики на тесте значительно ниже обучающих, модель переобучена.")

Задания 6-7
Train precision: 0.35
Train AUC: 0.97
Test precision: 0.04
Test AUC: 0.91
Вывод: метрики на тесте значительно ниже обучающих, модель переобучена.


##8. Улучшение рекомендаций.

Для этого можно провести:

1) настройку параметров модели LightFM;

2) добавить регуляризацию для борьбы с переобучением;

3) использовать жанры фильмов как признаки;

4) провести аугментацию данных, в частности добавить неявный фидбэк

(например, время просмотра, год выпуска и т.д.);

5) объединить несколько моделей для улучшения качества (например,

WARP, BPR, логистическую регрессию);

6) добавить текстовые описания фильмов.

Проанализируйте результат улучшенных рекомендаций, сделайте выводы.

In [ ]:
print("Задание 8")

optimized_lightfm_model = LightFM(
    loss='warp',
    item_alpha=1e-05,
    no_components=50
)

for improvement_epoch_index in range(30):
    optimized_lightfm_model.fit_partial(
        movielens_dataset['train'],
        item_features=movielens_dataset['item_features'],
        epochs=1,
        num_threads=2
    )

    if improvement_epoch_index % 10 == 0:
        print(f"Улучшение: Эпоха {improvement_epoch_index}")

optimized_train_precision = np.mean(lightfm_precision_at_k(
    optimized_lightfm_model,
    movielens_dataset['train'],
    item_features=movielens_dataset['item_features'],
    k=10,
    num_threads=2
))

optimized_test_precision = np.mean(lightfm_precision_at_k(
    optimized_lightfm_model,
    movielens_dataset['test'],
    item_features=movielens_dataset['item_features'],
    k=10,
    num_threads=2
))

print(f"Opt Tr P: {optimized_train_precision:.2f}")
print(f"Opt Te P: {optimized_test_precision:.2f}")
print("Вывод: модель демонстрирует критически низкую предсказательную способность на тестовой выборке. Регуляризация и добавление признаков объектов не устранили переобучение, разрыв между train и test метриками увеличился. Модель непригодна для эксплуатации.")

Задание 8
Улучшение: Эпоха 0
Улучшение: Эпоха 10
Улучшение: Эпоха 20
Opt Tr P: 0.57
Opt Te P: 0.03
Вывод: модель демонстрирует критически низкую предсказательную способность на тестовой выборке. Регуляризация и добавление признаков объектов не устранили переобучение, разрыв между train и test метриками увеличился. Модель непригодна для эксплуатации.
